Code:You CAPSTONE PROJECT: Survivor Analysis

In [1]:
import pandas as pd 


The data that is needed for this project is contained within the Castaways, Advantage Details, and Advantage Movement tabs of the survivor Excel sheet. Castaways contains information about the individuals who competed in each season, Advantage Details contains information about the advantages that appeared in each season, and Advantage Movement contains information about the what occurred with advantages during each season. 

In [2]:
# read in data

df_castaways = pd.read_excel('survivoR.xlsx', sheet_name="Castaways")
df_advantage_details = pd.read_excel('survivoR.xlsx', sheet_name="Advantage Details")
df_advantage_movement = pd.read_excel('survivoR.xlsx', sheet_name="Advantage Movement")

# Data Wrangling

## Function to Drop non-US rows

The scope of this project is limited to the United States version of Survivor. Each table that I am utilizing contains data for other versions that will need to removed. I created a function for efficiency. 

In [3]:
def drop_non_us (df: pd.DataFrame) -> pd.DataFrame:
    """Drops all rows from the table where the Version does not equal 'US' """
    df_non_US_dropped = df[df['Version']=="US"].copy()
    return df_non_US_dropped

## Clean and Feature Engineer Castaways

I began by cleaning the the Catsaways data.

In [4]:
# Castaways info

df_castaways.info()
df_castaways.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1403 entries, 0 to 1402
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Version         1403 non-null   object 
 1   Version Season  1403 non-null   object 
 2   Season          1403 non-null   int64  
 3   Full Name       1403 non-null   object 
 4   Castaway Id     1403 non-null   object 
 5   Castaway        1403 non-null   object 
 6   Age             1361 non-null   float64
 7   City            1349 non-null   object 
 8   State           1299 non-null   object 
 9   Episode         1361 non-null   float64
 10  Day             1361 non-null   float64
 11  Order           1361 non-null   float64
 12  Result          1361 non-null   object 
 13  Jury Status     601 non-null    object 
 14  Place           1403 non-null   int64  
 15  Original Tribe  1359 non-null   object 
 16  Jury            1403 non-null   bool   
 17  Finalist        1361 non-null   f

,Version,Version Season,Season,Full Name,Castaway Id,Castaway,Age,City,State,Episode,...,Jury,Finalist,Winner,Acknowledge,Ack Look,Ack Speak,Ack Gesture,Ack Smile,Ack Quote,Ack Score
0,AU,AU01,1,Des Quilty,AU0001,Des,59.0,Sunshine Coast,QLD,1.0,...,False,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AU,AU01,1,Bianca Anderson,AU0002,Bianca,36.0,Melbourne,VIC,2.0,...,False,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AU,AU01,1,Evan Jones,AU0003,Evan,30.0,Melbourne,VIC,3.0,...,False,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AU,AU01,1,Peter Fiegehen,AU0004,Peter,62.0,Canberra,ACT,4.0,...,False,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AU,AU01,1,Barry Lea,AU0005,Barry,44.0,Cairns,QLD,6.0,...,False,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


I dropped all of the columns that I was certain I wouldn't utilize in the analysis stage. I retained some columns that ultimately weren't utilized. 

In [5]:
# drop columns from Castaways that aren't needed

df_castaways.drop(columns=['City', 'State', 'Age', 'Episode', 'Day', 'Original Tribe', 'Acknowledge', 'Ack Look', 'Ack Speak', 'Ack Gesture', 'Ack Smile', 'Ack Quote', 'Ack Score', 'Castaway'], inplace=True)
print(df_castaways.columns)

Index(['Version', 'Version Season', 'Season', 'Full Name', 'Castaway Id',
       'Order', 'Result', 'Jury Status', 'Place', 'Jury', 'Finalist',
       'Winner'],
      dtype='object')


48 seasons have aired on tv as of the time of completing this project. Limited information has been released to the public about the competitors for the next two seasons of Survivor that will air, so they are included in the Castaways dataset. Because the rows for these seasons contain a large amount of null data and they do not have any corresponding data in the Advantage Movement and Advantage Details data, I filtered my dataframe so that it only includes data for season 48 and before.

In [6]:
# Drop seasons that haven't occurred yet
df_castaways = df_castaways[df_castaways['Season']<=48]

I renamed the columns to replace spaces with underscores and verified that the columns were renamed as expected.

In [7]:
# rename columns

df_castaways = df_castaways.rename(columns={'Version Season':'Version_Season', 'Full Name':'Full_Name', 'Castaway Id':'Castaway_Id', 'Jury Status':'Jury_Status'})
print(df_castaways.columns)

Index(['Version', 'Version_Season', 'Season', 'Full_Name', 'Castaway_Id',
       'Order', 'Result', 'Jury_Status', 'Place', 'Jury', 'Finalist',
       'Winner'],
      dtype='object')


I ensured that all of the columns were set to the correct datatype. 

In [8]:
# datatype

df_castaways= df_castaways.astype({'Version':'str', 'Version_Season':'str', 'Full_Name':'str', 'Castaway_Id':'str', 'Order':'int64','Result':'str', 'Jury_Status':'str', 'Finalist':'bool', 'Winner':'bool'})
print(df_castaways.dtypes)

Version           object
Version_Season    object
Season             int64
Full_Name         object
Castaway_Id       object
Order              int64
Result            object
Jury_Status       object
Place              int64
Jury                bool
Finalist            bool
Winner              bool
dtype: object


I combined the Version Season and Castaway Id columns to generate a unique ID for each row of data, which will serve as the primary key for this table in the sqlite database.. This will be utilized later for joining with another table.

In [9]:
#Create Season Castaway ID
df_castaways['Season_Castaway_Id'] = df_castaways['Version_Season'] + df_castaways['Castaway_Id']

I utilized my function to remove the rows for all versions other than the United States.

In [10]:
#check versions present in data 

print(df_castaways['Version'].unique())

#Drop non-US seasons and confirm that they were dropped

df_castaways_US = drop_non_us(df_castaways)
print(df_castaways_US['Version'].unique())

['AU' 'NZ' 'SA' 'UK' 'US']
['US']


In Survivor, someone's final standing can fall into four categories: Voted Out Pre-Jury, Jury, Finalist, and Winner. Partway through the season of Survivor the tribes merge. Anyone who was voted out prior to that time does not become a member of the jury. Each subsequent player who is voted out joins the jury until we reach the point of the final two or final three (depending on the season). Of the individuals who reach the final tribal council, one becomes The Sole Survivor, aka the winner. 

Because the goal of my project is to see how advantages impact who wins Survivor, I decided to categorize each player by their final standing, which I have referred to as their "finale categorization". 

The source data stored Winner, Finalist, and Jury as boolean values, but for the purposes of creating visualizations more easily, I combined that data into a single column containing a string. Anyone who was not a Winner, Finalist, or Jury was categorized as Voted Out Pre-Jury.

In [11]:
# Finale Categorization function

def finale_categorization (row: pd.Series) -> str:
        """ Takes the Jury, Finalist, and Winner columns and combine into one column that uses string instead of boolean """
        if row['Winner']:
              return 'Winner'
        elif row ['Finalist']:
              return 'Finalist'
        elif row['Jury']: 
            return 'Jury'
        else:
              return 'Voted Out Pre-Jury'


df_castaways_US['Finale_Categorization'] = df_castaways_US.apply(finale_categorization, axis=1)

finale_categorization_order = ['Voted Out Pre-Jury', 'Jury', 'Finalist', 'Winner']
df_castaways_US['Finale_Categorization'] = pd.Categorical(df_castaways_US['Finale_Categorization'], categories= finale_categorization_order, ordered=True)

print(df_castaways_US['Finale_Categorization'].value_counts())

Finale_Categorization
Jury                  402
Voted Out Pre-Jury    344
Finalist               81
Winner                 48
Name: count, dtype: int64


Survivor is generally considered to have "eras" in which the production of the show and style of game play have themes that lend them to be grouped together and distinguished from other eras. Because advantages did not exist with in Survivor when it debut and the ways in which advantages have been incorporated into gameplay have evolved over the years, I am going to categorize each season into an era. 

There are is no official categorization of these eras. For the purposes of this project, they will be grouped in the following manner based on a rough consensus among fans:
- Old School (pre-advantage):1-10
- Old School (with advantages): 11-20
- New School: 21-40
- New Era: 41 to present

In [12]:
# era categorization function

def era_categorization (season: int) -> str: 
    """ Categorize a season into an era based on its number """
    if season <=10:
        return 'Old School (pre-advantage)'
    elif season <=20:
        return 'Old School (with advantages)'
    elif season <=40:
        return 'New School'
    elif season >40:
        return 'New Era'
    else:
        return 'No Era Assigned'

df_castaways_US['Era'] = df_castaways_US['Season'].apply(era_categorization)

era_order = ['Old School (pre-advantage)', 'Old School (with advantages)', 'New School', 'New Era']
df_castaways_US['Era'] = pd.Categorical(df_castaways_US['Era'], categories=era_order, ordered=True)

df_castaways_US

,Version,Version_Season,Season,Full_Name,Castaway_Id,Order,Result,Jury_Status,Place,Jury,Finalist,Winner,Season_Castaway_Id,Finale_Categorization,Era
484,US,US01,1,Sonja Christopher,US0001,1,1st voted out,nan,16,False,False,False,US01US0001,Voted Out Pre-Jury,Old School (pre-advantage)
485,US,US01,1,B.B. Andersen,US0002,2,2nd voted out,nan,15,False,False,False,US01US0002,Voted Out Pre-Jury,Old School (pre-advantage)
486,US,US01,1,Stacey Stillman,US0003,3,3rd voted out,nan,14,False,False,False,US01US0003,Voted Out Pre-Jury,Old School (pre-advantage)
487,US,US01,1,Ramona Gray,US0004,4,4th voted out,nan,13,False,False,False,US01US0004,Voted Out Pre-Jury,Old School (pre-advantage)
488,US,US01,1,Dirk Been,US0005,5,5th voted out,nan,12,False,False,False,US01US0005,Voted Out Pre-Jury,Old School (pre-advantage)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1378,US,US48,48,Sai Hughley,US0729,7,7th voted out,nan,12,False,False,False,US48US0729,Voted Out Pre-Jury,New Era
1379,US,US48,48,Shauhin Davari,US0730,13,13th voted out,6th jury member,6,True,False,False,US48US0730,Jury,New Era
1380,US,US48,48,Star Toomey,US0731,11,11th voted out,4th jury member,8,True,False,False,US48US0731,Jury,New Era
1381,US,US48,48,Stephanie Berger,US0732,1,1st voted out,nan,18,False,False,False,US48US0732,Voted Out Pre-Jury,New Era


I did a final confirmation of what columns are in the dataframe before generating a csv file with the data, which will be used in the analysis notebook.

In [13]:
df_castaways_US.columns

Index(['Version', 'Version_Season', 'Season', 'Full_Name', 'Castaway_Id',
       'Order', 'Result', 'Jury_Status', 'Place', 'Jury', 'Finalist', 'Winner',
       'Season_Castaway_Id', 'Finale_Categorization', 'Era'],
      dtype='object')

In [14]:
df_castaways_US.to_csv('castaways_clean.csv', index=False)

## Clean and Feature Engineer Advantage Details

Next I cleaned the Advantage Details data.

In [15]:
df_advantage_details.info()
df_advantage_details.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Version         395 non-null    object
 1   Version Season  395 non-null    object
 2   Season          395 non-null    int64 
 3   Advantage Id    395 non-null    int64 
 4   Advantage Type  395 non-null    object
 5   Clue Details    395 non-null    object
 6   Location Found  387 non-null    object
 7   Conditions      194 non-null    object
dtypes: int64(2), object(6)
memory usage: 24.8+ KB


,Version,Version Season,Season,Advantage Id,Advantage Type,Clue Details,Location Found,Conditions
0,US,US11,11,1,Preventative Hidden Immunity Idol,Found without clue,Found around camp,Play before votes are cast
1,US,US12,12,1,Super Idol,Found on Exile,Advantage from Exile,Play after votes are read; Valid until F4
2,US,US13,13,1,Super Idol,Found on Exile,Advantage from Exile,Play after votes are read; Valid until F4
3,US,US14,14,1,Hidden Immunity Idol,Found on Exile,Found around camp,NaN
4,US,US14,14,2,Hidden Immunity Idol,Someone shared the clue,Found around camp,NaN


I renamed the columns to replace spaces with underscores and verified that the columns were renamed as expected.

In [16]:
#rename columns

df_advantage_details = df_advantage_details.rename(columns={'Version Season':'Version_Season', 'Advantage Id':'Advantage_Id', 'Advantage Type':'Advantage_Type', 'Clue Details':'Clue_Details','Location Found':'Location_Found'})
print(df_advantage_details.columns)

Index(['Version', 'Version_Season', 'Season', 'Advantage_Id', 'Advantage_Type',
       'Clue_Details', 'Location_Found', 'Conditions'],
      dtype='object')


I ensured that all of the columns were set to the correct datatype.

In [17]:
# Ensure all columns have the correct data type

df_advantage_details= df_advantage_details.astype({'Version_Season':'str', 'Advantage_Id':'str', 'Advantage_Type':'str', 'Clue_Details':'str', 'Location_Found':'str', 'Conditions':'str'})
print(df_advantage_details.dtypes)

Version           object
Version_Season    object
Season             int64
Advantage_Id      object
Advantage_Type    object
Clue_Details      object
Location_Found    object
Conditions        object
dtype: object


I combined the Version Season and Advantage Id columns to generate a unique ID for each row of data, which will serve as the primary key for this table in the sqlite database. This will be utilized later for joining with another table.

In [18]:
#Create Season Advantage ID column by combining Version Season and Advantage Id

df_advantage_details['Season_Advantage_Id']= df_advantage_details['Version_Season']+df_advantage_details['Advantage_Id']
df_advantage_details.columns

Index(['Version', 'Version_Season', 'Season', 'Advantage_Id', 'Advantage_Type',
       'Clue_Details', 'Location_Found', 'Conditions', 'Season_Advantage_Id'],
      dtype='object')

I dropped all rows of data for any version but United States.

In [19]:
#drop non-US rows

df_advantage_details_US = drop_non_us(df_advantage_details)
print(df_advantage_details_US['Version'].unique())

['US']


I generated a csv file with the data from the dataframe, which will be used in the analysis notebook.

In [20]:
df_advantage_details_US.to_csv('advantage_details_clean.csv', index = False)

## Clean and Feature Engineer Advantage Movement

Lastly, I cleaned the Advantage Movement data.

In [21]:
#Advantage Movement info

df_advantage_movement.info()
df_advantage_movement.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 902 entries, 0 to 901
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Version          902 non-null    object 
 1   Version Season   902 non-null    object 
 2   Season           902 non-null    int64  
 3   Castaway         902 non-null    object 
 4   Castaway Id      902 non-null    object 
 5   Advantage Id     902 non-null    int64  
 6   Sequence Id      902 non-null    int64  
 7   Day              885 non-null    float64
 8   Episode          885 non-null    float64
 9   Event            902 non-null    object 
 10  Played for       274 non-null    object 
 11  Played for Id    273 non-null    object 
 12  Success          276 non-null    object 
 13  Votes Nullified  206 non-null    float64
 14  Sog Id           881 non-null    float64
dtypes: float64(4), int64(3), object(8)
memory usage: 105.8+ KB


,Version,Version Season,Season,Castaway,Castaway Id,Advantage Id,Sequence Id,Day,Episode,Event,Played for,Played for Id,Success,Votes Nullified,Sog Id
0,US,US11,11,Gary,US0161,1,1,24.0,9.0,Found,NaN,NaN,NaN,NaN,10.0
1,US,US11,11,Gary,US0161,1,2,24.0,9.0,Played,Gary,US0161,Yes,0.0,10.0
2,US,US12,12,Terry,US0180,1,1,9.0,4.0,Found,NaN,NaN,NaN,NaN,4.0
3,US,US12,12,Terry,US0180,1,2,37.0,15.0,Expired,NaN,NaN,NaN,NaN,13.0
4,US,US13,13,Yul,US0202,1,1,5.0,2.0,Found,NaN,NaN,NaN,NaN,2.0


I dropped all of the columns that I was certain I wouldn't utilize in the analysis stage. I retained some columns that ultimately weren't utilized.

In [22]:
#drop columns from Advantage Movement that aren't needed for this analysis

df_advantage_movement.drop(columns=['Castaway','Day', 'Episode', 'Votes Nullified', 'Played for', 'Played for Id', 'Sog Id'], inplace=True)
print(df_advantage_movement.columns)

Index(['Version', 'Version Season', 'Season', 'Castaway Id', 'Advantage Id',
       'Sequence Id', 'Event', 'Success'],
      dtype='object')


I renamed the columns to replace spaces with underscores and verified that the columns were renamed as expected.

In [23]:
#rename columns
df_advantage_movement = df_advantage_movement.rename(columns={'Version Season':'Version_Season', 'Castaway Id':'Castaway_Id', 'Advantage Id':'Advantage_Id', 'Sequence Id':'Sequence_Id'})
print(df_advantage_movement.columns)

Index(['Version', 'Version_Season', 'Season', 'Castaway_Id', 'Advantage_Id',
       'Sequence_Id', 'Event', 'Success'],
      dtype='object')



I ensured that all of the columns were set to the correct datatype.

In [24]:
# Ensure all columns have the correct data type
df_advantage_movement = df_advantage_movement.astype({'Version':'str', 'Version_Season':'str', 'Castaway_Id':'str', 'Advantage_Id':'str', 'Event':'str', 'Success':'str'})
df_advantage_movement.dtypes
                                                      

Version           object
Version_Season    object
Season             int64
Castaway_Id       object
Advantage_Id      object
Sequence_Id        int64
Event             object
Success           object
dtype: object

I combined the Version Season and Advantage Id fields to generate column that can be joined to the Advantage Details table.

In [25]:
#Create Season Advantage ID column by combining Version Season and Advantage Id

df_advantage_movement['Season_Advantage_Id'] = df_advantage_movement['Version_Season']+df_advantage_movement['Advantage_Id']
df_advantage_movement.head()

,Version,Version_Season,Season,Castaway_Id,Advantage_Id,Sequence_Id,Event,Success,Season_Advantage_Id
0,US,US11,11,US0161,1,1,Found,nan,US111
1,US,US11,11,US0161,1,2,Played,Yes,US111
2,US,US12,12,US0180,1,1,Found,nan,US121
3,US,US12,12,US0180,1,2,Expired,nan,US121
4,US,US13,13,US0202,1,1,Found,nan,US131


I combined the Version Season and Castaway Id columns to generate a column that can be joined to the Castaways table. 

In [26]:
# Create Season Castaway ID column by combining Version Season and Castaway Id

df_advantage_movement['Season_Castaway_Id'] = df_advantage_movement['Version_Season']+df_advantage_movement['Castaway_Id']
df_advantage_movement.head()

,Version,Version_Season,Season,Castaway_Id,Advantage_Id,Sequence_Id,Event,Success,Season_Advantage_Id,Season_Castaway_Id
0,US,US11,11,US0161,1,1,Found,nan,US111,US11US0161
1,US,US11,11,US0161,1,2,Played,Yes,US111,US11US0161
2,US,US12,12,US0180,1,1,Found,nan,US121,US12US0180
3,US,US12,12,US0180,1,2,Expired,nan,US121,US12US0180
4,US,US13,13,US0202,1,1,Found,nan,US131,US13US0202


I created a column that begins at 1 and increments up by 1 each row in order to create a unique ID for each row of data. This will be utilized as the primary key in the sqlite database. 

In [27]:
#Create an id column to serve as primary key

df_advantage_movement['Movement_Id'] = range(1, len(df_advantage_movement)+1)
df_advantage_movement.columns

Index(['Version', 'Version_Season', 'Season', 'Castaway_Id', 'Advantage_Id',
       'Sequence_Id', 'Event', 'Success', 'Season_Advantage_Id',
       'Season_Castaway_Id', 'Movement_Id'],
      dtype='object')

I dropped the rows for any non-US version.

In [28]:
#Drop non-US rows

df_advantage_movement_US = drop_non_us(df_advantage_movement)
print(df_advantage_movement_US['Version'].unique())

['US']


I generated a csv file to be used in the analysis workbook.

In [29]:
df_advantage_movement_US.to_csv('advantage_movement_clean.csv', index = False)